In [ ]:
!pip install -q -U transformers peft trl bitsandbytes datasets accelerate torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.0/925.0 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 77.0 MB/s eta 0:00:00


In [2]:
!git clone https://github.com/berraEgcin/data_handling_project_26.git
%cd data_handling_project_26
!git checkout feature/sailesh-phase2-model-pipeline

Cloning into 'data_handling_project_26'...
remote: Enumerating objects: 409, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 409 (delta 0), reused 0 (delta 0), pack-reused 402 (from 1)
Receiving objects: 100% (409/409), 1.62 GiB | 15.08 MiB/s, done.
Resolving deltas: 100% (229/229), done.
/content/data_handling_project_26
branch 'feature/sailesh-phase2-model-pipeline' set up to track 'origin/feature/sailesh-phase2-model-pipeline'.
Switched to a new branch 'feature/sailesh-phase2-model-pipeline'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import torch
from google.colab import userdata
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

# Load HF Token from Secrets if available
try:
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    print("HF_TOKEN loaded from Colab Secrets.")
except Exception:
    pass

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
TRAIN_DATA_PATH = "data/processed/train.jsonl"
VAL_DATA_PATH = "data/processed/val.jsonl"
OUTPUT_DIR = "/content/drive/MyDrive/clinical_lora_adapter"

def format_prompts(batch):
    formatted_texts = []
    for inp, out in zip(batch["input"], batch["output"]):
        text = (
            f"<|im_start|>system\nYou are a clinical lab interpreter. Extract abnormal findings as valid JSON.<|im_end|>\n"
            f"<|im_start|>user\n{inp}<|im_end|>\n"
            f"<|im_start|>assistant\n{out}<|im_end|>"
        )
        formatted_texts.append(text)
    return {"text": formatted_texts}

print("Loading data...")
dataset = load_dataset("json", data_files={"train": TRAIN_DATA_PATH, "validation": VAL_DATA_PATH})
dataset = dataset.map(format_prompts, batched=True)

# 1. 4-bit Quantization Config (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# 2. Tokenizer & Base Model
print(f"Loading base model: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Force any non-quantized residual float parameters to float32 to prevent GradScaler conflicts
for name, module in model.named_modules():
    if "norm" in name or "lm_head" in name:
        module.to(torch.float32)

# 3. LoRA Configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

# 4. SFTConfig: disable fp16 auto-scaling since QLoRA computes in 4-bit/fp16 under the hood
training_args = SFTConfig(
    output_dir="./tmp_checkpoints",
    dataset_text_field="text",
    max_length=512,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    eval_strategy="epoch",
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    fp16=False,
    bf16=False,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    report_to="none",
)

# 5. Initialize Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)

print("Starting training...")
trainer.train()

print(f"Saving adapter weights to {OUTPUT_DIR}...")
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Training complete!")

Loading data...


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/931 [00:00<?, ? examples/s]

Map:   0%|          | 0/77 [00:00<?, ? examples/s]

...

Dropping fully masked examples from eval dataset:   0%|          | 0/77 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting training...


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

In [ ]:
import os
import re
import json
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from sklearn.metrics import classification_report, precision_recall_fscore_support

# 1. Set paths
REPO_DIR = "/content/data_handling_project_26"
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)

BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = "/content/drive/MyDrive/clinical_lora_adapter"
TEST_FILE = "data/processed/test.jsonl"

# 2. Load 4-bit Base Model and LoRA Adapter from Drive
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer and base model...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    dtype=torch.float16,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Loading LoRA weights from {ADAPTER_PATH}...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print("Model loaded successfully!")

# 3. Read unseen test records
test_samples = []
with open(TEST_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            test_samples.append(json.loads(line))

print(f"Loaded {len(test_samples)} test samples. Running batch evaluation...")

# 4. Batch Inference
eval_pairs = []
valid_json_count = 0

for idx, item in enumerate(test_samples):
    prompt = (
        f"<|im_start|>system\nYou are a clinical lab interpreter. Extract abnormal findings as valid JSON.<|im_end|>\n"
        f"<|im_start|>user\n{item['input']}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    pred_str = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    try:
        json_match = re.search(r"\{.*\}", pred_str, re.DOTALL)
        pred_dict = json.loads(json_match.group()) if json_match else {}
        valid_json_count += 1
    except Exception:
        pred_dict = {"abnormal_findings": [], "interpretation_summary": ""}

    gold_dict = json.loads(item["output"])
    eval_pairs.append((gold_dict, pred_dict))

    if (idx + 1) % 10 == 0 or (idx + 1) == len(test_samples):
        print(f"Evaluated {idx + 1}/{len(test_samples)} samples...")

# 5. Compute Scikit-Learn Metrics
ALL_PARAMETERS = [
    "Glucose",
    "Urea Nitrogen",
    "Calcium",
    "Sodium",
    "Potassium",
    "Chloride",
    "Carbon Dioxide",
    "HbA1c"
]

y_true = []
y_pred = []

for gold, pred in eval_pairs:
    gold_set = {f["parameter"] for f in gold.get("abnormal_findings", [])}
    pred_set = {
        f.get("parameter")
        for f in pred.get("abnormal_findings", [])
        if isinstance(f, dict) and "parameter" in f
    }

    for param in ALL_PARAMETERS:
        y_true.append(1 if param in gold_set else 0)
        y_pred.append(1 if param in pred_set else 0)

precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary")
json_rate = (valid_json_count / len(test_samples)) * 100

summary_df = pd.DataFrame([{
    "Test Dataset": "test.jsonl",
    "Total Samples": len(test_samples),
    "Valid JSON Syntax": f"{json_rate:.2f}%",
    "Precision": f"{precision:.4f}",
    "Recall": f"{recall:.4f}",
    "F1-Score": f"{f1:.4f}"
}])

print("\n" + "=" * 60)
print("             SPRINT 1: FINAL MODEL METRICS TABLE")
print("=" * 60)
display(summary_df)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["Normal (0)", "Abnormal (1)"]))

# 6. Save metrics payload
metrics_export = {
    "total_samples": len(test_samples),
    "json_validity_rate_pct": round(json_rate, 2),
    "precision": round(precision, 4),
    "recall": round(recall, 4),
    "f1_score": round(f1, 4)
}

with open("data/processed/sprint1_evaluation_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_export, f, indent=2)

print("\nSaved metrics to data/processed/sprint1_evaluation_metrics.json")

Loading tokenizer and base model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading LoRA weights from /content/drive/MyDrive/clinical_lora_adapter...


/usr/local/lib/python3.13/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['kasa_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Model loaded successfully!
Loaded 78 test samples. Running batch evaluation...
Evaluated 10/78 samples...
Evaluated 20/78 samples...
Evaluated 30/78 samples...
Evaluated 40/78 samples...
Evaluated 50/78 samples...
Evaluated 60/78 samples...
Evaluated 70/78 samples...
Evaluated 78/78 samples...

             SPRINT 1: FINAL MODEL METRICS TABLE


,Test Dataset,Total Samples,Valid JSON Syntax,Precision,Recall,F1-Score
0,test.jsonl,78,100.00%,0.7442,0.6957,0.7191



Classification Report:
              precision    recall  f1-score   support

  Normal (0)       0.95      0.96      0.95       532
Abnormal (1)       0.74      0.70      0.72        92

    accuracy                           0.92       624
   macro avg       0.85      0.83      0.84       624
weighted avg       0.92      0.92      0.92       624


Saved metrics to data/processed/sprint1_evaluation_metrics.json


In [ ]:
!pip install -q chromadb sentence-transformers scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 106.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [ ]:
# -------------------------------------------------------------
# Sprint 2 - Activity 1: Clinical Knowledge Base Definition
# -------------------------------------------------------------
CLINICAL_KNOWLEDGE_BASE = [
    {
        "id": "ref_glucose",
        "parameter": "Glucose",
        "content": "Clinical Reference for Glucose: Normal fasting range is 70.0 to 99.0 mg/dL. Values below 70.0 mg/dL indicate Hypoglycemia (Low). Values 100.0 mg/dL or higher indicate Hyperglycemia or impaired fasting glucose (High)."
    },
    {
        "id": "ref_bun",
        "parameter": "Urea Nitrogen",
        "content": "Clinical Reference for Urea Nitrogen (BUN): Normal range is 7.0 to 20.0 mg/dL. Values below 7.0 mg/dL indicate Low BUN. Values above 20.0 mg/dL indicate elevated BUN or renal impairment (High)."
    },
    {
        "id": "ref_calcium",
        "parameter": "Calcium",
        "content": "Clinical Reference for Total Calcium: Normal range is 8.5 to 10.2 mg/dL. Values below 8.5 mg/dL indicate Hypocalcemia (Low). Values above 10.2 mg/dL indicate Hypercalcemia (High)."
    },
    {
        "id": "ref_sodium",
        "parameter": "Sodium",
        "content": "Clinical Reference for Sodium: Normal serum electrolyte range is 135.0 to 145.0 mmol/L. Values below 135.0 mmol/L indicate Hyponatremia (Low). Values above 145.0 mmol/L indicate Hypernatremia (High)."
    },
    {
        "id": "ref_potassium",
        "parameter": "Potassium",
        "content": "Clinical Reference for Potassium: Normal serum electrolyte range is 3.5 to 5.0 mmol/L. Values below 3.5 mmol/L indicate Hypokalemia (Low). Values above 5.0 mmol/L indicate Hyperkalemia (High)."
    },
    {
        "id": "ref_chloride",
        "parameter": "Chloride",
        "content": "Clinical Reference for Chloride: Normal serum electrolyte range is 96.0 to 106.0 mmol/L. Values below 96.0 mmol/L indicate Hypochloremia (Low). Values above 106.0 mmol/L indicate Hyperchloremia (High)."
    },
    {
        "id": "ref_co2",
        "parameter": "Carbon Dioxide",
        "content": "Clinical Reference for Carbon Dioxide (Bicarbonate): Normal range is 22.0 to 29.0 mmol/L. Values below 22.0 mmol/L indicate metabolic acidosis or Low CO2. Values above 29.0 mmol/L indicate metabolic alkalosis (High)."
    },
    {
        "id": "ref_hba1c",
        "parameter": "HbA1c",
        "content": "Clinical Reference for Glycated Hemoglobin (HbA1c): Normal reference range is below 5.7%. Values from 5.7% to 6.4% indicate Prediabetes (High). Values 6.5% or higher indicate Diabetes Mellitus (High)."
    }
]

doc_ids = [doc["id"] for doc in CLINICAL_KNOWLEDGE_BASE]
documents = [doc["content"] for doc in CLINICAL_KNOWLEDGE_BASE]
print(f"Loaded {len(documents)} clinical reference documents into knowledge base.")

Loaded 8 clinical reference documents into knowledge base.


In [ ]:
# -------------------------------------------------------------
# Sprint 2 - Activity 2: TF-IDF Classical Retrieval Baseline
# -------------------------------------------------------------
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

def retrieve_with_tfidf(query_text: str, top_k: int = 3):
    """Retrieves top-k documents using sparse TF-IDF vectors and cosine similarity."""
    query_vec = tfidf_vectorizer.transform([query_text])
    sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = np.argsort(sim_scores)[::-1][:top_k]
    return [documents[i] for i in top_indices if sim_scores[i] > 0]

# Verification test query
sample_patient_query = "Glucose: 104.6 mg/dL, Carbon Dioxide: 20.6 mmol/L"
print("TF-IDF Retrieved References:")
for ref in retrieve_with_tfidf(sample_patient_query, top_k=2):
    print("-", ref)

In [ ]:
# -------------------------------------------------------------
# Sprint 2 - Activity 3: Modern Vector DB (ChromaDB) + Embeddings
# -------------------------------------------------------------
import torch
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

# 1. Initialize local in-memory ChromaDB client
chroma_client = chromadb.Client()

# 2. Configure embedding function using all-MiniLM-L6-v2
embedding_fn = SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# 3. Create or reset collection
collection_name = "clinical_lab_reference_db"
try:
    chroma_client.delete_collection(name=collection_name)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=collection_name,
    embedding_function=embedding_fn
)

# 4. Insert reference documents into Vector DB
collection.add(
    ids=doc_ids,
    documents=documents
)

print(f"Successfully indexed {collection.count()} reference ranges into ChromaDB.")

def retrieve_with_dense_rag(query_text: str, top_k: int = 3):
    """Queries ChromaDB to fetch semantically relevant reference ranges."""
    results = collection.query(
        query_texts=[query_text],
        n_results=top_k
    )
    return results["documents"][0]

sample_patient_query = "Glucose: 104.6 mg/dL, Carbon Dioxide: 20.6 mmol/L"
print("\nChromaDB Dense RAG Retrieved References:")
for ref in retrieve_with_dense_rag(sample_patient_query, top_k=2):
    print("-", ref)

In [ ]:
# -------------------------------------------------------------
# Sprint 2 - Activity 4: RAG vs Fine-Tuning Inference Test
# -------------------------------------------------------------
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = "/content/drive/MyDrive/clinical_lora_adapter"

print("Loading tokenizer and base model in float16 directly...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Load directly in 16-bit precision (fits easily within Colab T4 GPU VRAM)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Loading LoRA weights from {ADAPTER_PATH}...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print("Model loaded successfully!")

# Test case with borderline Carbon Dioxide: 20.6 mmol/L
borderline_patient_case = (
    "Patient Demographics: Gender=M. Lab Results: Glucose: 104.6 mg/dL, "
    "Urea Nitrogen: 12.7 mg/dL, Calcium: 10.1 mg/dL, Sodium: 136.9 mmol/L, "
    "Potassium: 4.3 mmol/L, Chloride: 107.8 mmol/L, Carbon Dioxide: 20.6 mmol/L, HbA1c: 4.4 %"
)

# 1. Retrieve clinical reference context via ChromaDB
retrieved_docs = retrieve_with_dense_rag(borderline_patient_case, top_k=3)
context_str = "\n".join([f"- {doc}" for doc in retrieved_docs])

# 2. Build Grounded RAG Prompt
rag_prompt = (
    f"<|im_start|>system\nYou are a clinical lab interpreter. Extract abnormal findings as valid JSON with keys "
    f"'abnormal_findings' and 'interpretation_summary'. Use the provided Reference Guidelines to evaluate borderline values accurately.\n\n"
    f"Clinical Reference Guidelines:\n{context_str}<|im_end|>\n"
    f"<|im_start|>user\n{borderline_patient_case}<|im_end|>\n"
    f"<|im_start|>assistant\n"
)

# 3. Model generation with RAG context
inputs = tokenizer(rag_prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

rag_prediction = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

print("\n" + "=" * 60)
print("=== RETRIEVED CONTEXT INJECTED ===")
print(context_str)
print("\n=== MODEL OUTPUT WITH RAG CONTEXT ===")
print(rag_prediction)

Loading tokenizer and base model in float16 directly...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading LoRA weights from /content/drive/MyDrive/clinical_lora_adapter...


/usr/local/lib/python3.13/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['kasa_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [ ]:
# Test with top_k=8 so all relevant lab reference guidelines are in context
retrieved_docs_all = retrieve_with_dense_rag(borderline_patient_case, top_k=8)
context_all_str = "\n".join([f"- {doc}" for doc in retrieved_docs_all])

rag_all_prompt = (
    f"<|im_start|>system\nYou are a clinical lab interpreter. Extract abnormal findings as valid JSON with keys "
    f"'abnormal_findings' and 'interpretation_summary'. Use the provided Reference Guidelines to evaluate borderline values accurately.\n\n"
    f"Clinical Reference Guidelines:\n{context_all_str}<|im_end|>\n"
    f"<|im_start|>user\n{borderline_patient_case}<|im_end|>\n"
    f"<|im_start|>assistant\n"
)

inputs = tokenizer(rag_all_prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

print("=== MODEL OUTPUT WITH COMPLETE RAG CONTEXT (top_k=8) ===")
print(tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip())

=== MODEL OUTPUT WITH COMPLETE RAG CONTEXT (top_k=8) ===
{"abnormal_findings": [{"parameter": "Glucose", "value": 104.6, "unit": "mg/dL", "flag": "High"}, {"parameter": "Chloride", "value": 107.8, "unit": "mmol/L", "flag": "High"}], "interpretation_summary": "Identified 2 abnormal parameter(s)."}


In [ ]:
strong_rag_prompt = (
    f"<|im_start|>system\n"
    f"You are a clinical lab interpreter. Follow these steps:\n"
    f"1. Compare every single lab value strictly against the Clinical Reference Guidelines below.\n"
    f"2. Pay special attention to Carbon Dioxide (normal: 22.0 - 29.0 mmol/L). Any value below 22.0 is Low.\n"
    f"3. Return valid JSON containing 'abnormal_findings' and 'interpretation_summary'.\n\n"
    f"Clinical Reference Guidelines:\n{context_all_str}<|im_end|>\n"
    f"<|im_start|>user\n{borderline_patient_case}<|im_end|>\n"
    f"<|im_start|>assistant\n"
)

inputs = tokenizer(strong_rag_prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

print("=== STRONG GROUNDED RAG OUTPUT ===")
print(tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip())

=== STRONG GROUNDED RAG OUTPUT ===
{"abnormal_findings": [{"parameter": "Glucose", "value": 104.6, "unit": "mg/dL", "flag": "High"}, {"parameter": "Chloride", "value": 107.8, "unit": "mmol/L", "flag": "High"}], "interpretation_summary": "Identified 2 abnormal parameter(s)."}


In [ ]:
%%writefile /content/data_handling_project_26/app.py
import streamlit as st
import json
import torch
import chromadb
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

st.set_page_config(page_title="Clinical Lab AI Assistant", layout="wide")

st.title("🩺 Clinical Lab Interpretation Assistant")
st.markdown("Modern AI pipeline: **Qwen2.5-1.5B-Instruct + LoRA + ChromaDB RAG**")

# 1. Setup & Cache Model
@st.cache_resource
def load_clinical_pipeline():
    base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
    adapter_path = "/content/drive/MyDrive/clinical_lora_adapter"

    tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()

    chroma_client = chromadb.Client()
    collection = chroma_client.get_or_create_collection(name="clinical_reference_app")

    references = [
        ("ref_glucose", "Clinical Reference for Glucose: Normal fasting range is 70.0 to 99.0 mg/dL."),
        ("ref_bun", "Clinical Reference for Urea Nitrogen (BUN): Normal range is 7.0 to 20.0 mg/dL."),
        ("ref_calcium", "Clinical Reference for Total Calcium: Normal range is 8.5 to 10.2 mg/dL."),
        ("ref_sodium", "Clinical Reference for Sodium: Normal range is 135.0 to 145.0 mmol/L."),
        ("ref_potassium", "Clinical Reference for Potassium: Normal range is 3.5 to 5.0 mmol/L."),
        ("ref_chloride", "Clinical Reference for Chloride: Normal range is 96.0 to 106.0 mmol/L."),
        ("ref_co2", "Clinical Reference for Carbon Dioxide (CO2): Normal range is 22.0 to 29.0 mmol/L."),
        ("ref_hba1c", "Clinical Reference for Glycated Hemoglobin (HbA1c): Normal range is below 5.7%.")
    ]
    if collection.count() == 0:
        collection.add(
            ids=[r[0] for r in references],
            documents=[r[1] for r in references]
        )

    return tokenizer, model, collection

with st.spinner("Loading base model, LoRA adapters, and ChromaDB vector store..."):
    tokenizer, model, collection = load_clinical_pipeline()

# 2. Main Layout
col1, col2 = st.columns([1, 1])

default_case = (
    "Patient Demographics: Gender=M. Lab Results: Glucose: 104.6 mg/dL, "
    "Urea Nitrogen: 12.7 mg/dL, Calcium: 10.1 mg/dL, Sodium: 136.9 mmol/L, "
    "Potassium: 4.3 mmol/L, Chloride: 107.8 mmol/L, Carbon Dioxide: 20.6 mmol/L, HbA1c: 4.4 %"
)

with col1:
    st.subheader("Patient Input")
    patient_query = st.text_area("Enter Patient Demographics & Lab Results:", value=default_case, height=140)
    top_k = st.slider("ChromaDB Top-K Retrieved Guidelines", min_value=1, max_value=8, value=3)
    run_btn = st.button("Interpret Lab Values", type="primary")

with col2:
    st.subheader("RAG Retrieved Context")
    if patient_query:
        search_res = collection.query(query_texts=[patient_query], n_results=top_k)
        retrieved_docs = search_res["documents"][0] if search_res["documents"] else []
        for doc in retrieved_docs:
            st.info(f"📌 {doc}")

if run_btn:
    st.markdown("---")
    st.subheader("Model Diagnostic Output")

    context_str = "\n".join([f"- {d}" for d in retrieved_docs])
    rag_prompt = (
        f"<|im_start|>system\nYou are a clinical lab interpreter. Extract abnormal findings as valid JSON with keys "
        f"'abnormal_findings' and 'interpretation_summary'.\n\n"
        f"Clinical Reference Guidelines:\n{context_str}<|im_end|>\n"
        f"<|im_start|>user\n{patient_query}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    inputs = tokenizer(rag_prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    pred_str = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    try:
        parsed = json.loads(pred_str)
        st.success("JSON Syntax Validated ✅")
        st.json(parsed)
    except Exception:
        st.warning("Raw Generation Output:")
        st.code(pred_str)

Writing /content/data_handling_project_26/app.py


In [ ]:
!pip install -q streamlit
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 85.6 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏
added 22 packages in 3s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏

In [ ]:
import urllib.request

external_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print("=" * 60)
print("YOUR TUNNEL PASSWORD (IP):", external_ip)
print("=" * 60)

!streamlit run /content/data_handling_project_26/app.py & npx localtunnel --port 8501

YOUR TUNNEL PASSWORD (IP): 34.139.1.79
⠙⠹⠸⠼⠴⠦⠧⠇⠏your url is: https://public-pears-stare.loca.lt


2026-09-29 07:17:32.874 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.139.1.79:8501



In [ ]:
!streamlit run /content/data_handling_project_26/app.py \
    --server.enableCORS=false \
    --server.enableXsrfProtection=false \
    --server.port=8501 & npx localtunnel --port 8501

In [ ]:
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null
!cloudflared tunnel --url http://localhost:8501

2026-09-28T20:09:16Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-28T20:09:16Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-28T20:09:19Z INF +--------------------------------------------------------------------------------------------+
2026-09-28T20:09:19Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-28T20:09:19Z INF |  https://abstracts-ron-hidden-supporting.trycloudflare

In [ ]:
import time
from google.colab import output

# 1. Start Streamlit in the background
get_ipython().system_raw(
    "streamlit run /content/data_handling_project_26/app.py "
    "--server.enableCORS=false "
    "--server.enableXsrfProtection=false "
    "--server.port=8501 &"
)

# 2. Wait 3 seconds for the server to bind
time.sleep(3)

# 3. Embed the running Streamlit UI directly inside your notebook
output.serve_kernel_port_as_iframe(8501, width="100%", height="750")

<IPython.core.display.Javascript object>